# High-rise 05 - Facade Cp profile

Compute per-triangle Cp statistics (mean / min / max over the record) and
sample the mean field down a vertical line on one facade for a
height-resolved pressure profile -- the reliable, always-headless
deliverable. A per-triangle mesh-field image renderer previously lived
here; it produced illegible output for tall/slender buildings and has been
removed pending a proper flattened 2-D facade projection. If the optional
`[vtk]` extra (PyVista) is installed, a contoured, colour-barred whole-body
snapshot is written to `debug/` as an extra.

In [ ]:
import os
import pathlib

import numpy as np  # noqa: F401  (used across later cells)

import matplotlib

matplotlib.use("Agg")  # headless: notebooks write files, they do not display

from cfdmod import inflow_report, mesh_field, plot_config  # noqa: E402
from cfdmod.building import (  # noqa: E402
    BuildingCase,
    cf_per_floor,
    cm_per_floor,
    cp_from_pressure,
    example_building_case,
    example_building_structure,
    floor_accelerations,
    floor_load_source,
    peak_response_table,
    peak_value,
    plot_floor_mass,
    plot_mode_shape,
    plot_natural_frequencies,
    solve_building_response,
    structure_from_csvs,
)
from cfdmod.report import DebugWriter  # noqa: E402


def _find_repo(start: pathlib.Path) -> pathlib.Path:
    p = start.resolve()
    while p != p.parent:
        if (p / "pyproject.toml").exists():
            return p
        p = p.parent
    return start.resolve()


REPO = _find_repo(pathlib.Path.cwd())

plot_config.apply_style()

FIX = REPO / "fixtures" / "tests"
OUTPUT_BASE = pathlib.Path(
    os.environ.get("CFDMOD_HR_OUTPUT_BASE", REPO / "examples" / "high_rise" / "_run")
)
VERSION = os.environ.get("CFDMOD_HR_VERSION", "example")
print("REPO:", REPO)
print("OUTPUT_BASE:", OUTPUT_BASE, "| VERSION:", VERSION)

In [ ]:
from cfdmod.adapters.xdmf_h5 import XdmfH5Storage

# --- config -------------------------------------------------------------
MESH = pathlib.Path(
    os.environ.get("CFDMOD_HR_MESH", FIX / "pressure" / "galpao" / "galpao.normalized.lnas")
)
ARTIFACTS = OUTPUT_BASE / "artifacts" / VERSION
cp = XdmfH5Storage(ARTIFACTS).read_data_source(pathlib.Path("cp.time_series"))

geom = mesh_field.load_geometry(MESH)
cp_series = np.asarray(cp.fields.read("cp"))
cp_stats = {
    "mean": np.nanmean(cp_series, axis=1),
    "min": np.nanmin(cp_series, axis=1),
    "max": np.nanmax(cp_series, axis=1),
}
print("cp stats over", cp_stats["mean"].shape[0], "triangles")

In [ ]:
import pandas as pd

# --- deliverable ----------------------------------------------------------
dbg = DebugWriter(OUTPUT_BASE, stage="facade", version=VERSION)
summary = pd.DataFrame(
    {stat: [float(np.nanmin(vals)), float(np.nanmean(vals)), float(np.nanmax(vals))]
     for stat, vals in cp_stats.items()},
    index=["min", "mean", "max"],
)
dbg.save_csv(summary.reset_index(names="of"), "cp_stats_summary.csv", deliverable=True)

# Optional high-quality PyVista render (only if the [vtk] extra is installed).
vtp = dbg.debug_path("cp_body.vtp")
if mesh_field.write_field_vtp(geom, {"Cp_mean": cp_stats["mean"]}, vtp):
    clim = (float(np.nanmin(cp_stats["min"])), float(np.nanmax(cp_stats["max"])))
    ok = mesh_field.render_vtp_snapshot(
        vtp, dbg.debug_path("cp_mean_pyvista.png"),
        scalar="Cp_mean", label="mean Cp", clim=clim,
    )
    print("pyvista snapshot:", ok)
else:
    print("pyvista/[vtk] not installed - skipping the optional snapshot")
print("summary ->", dbg.deliverables_dir)

In [ ]:
# --- facade pressure profile along a vertical line ----------------------
# Sample the mean-Cp field down a vertical line on the front (-y) facade.
verts = np.asarray(geom.vertices)
vmin = verts.min(axis=0)
vmax = verts.max(axis=0)
xc = float((vmin[0] + vmax[0]) / 2)
p1 = (xc, float(vmin[1]), float(vmin[2]))
p2 = (xc, float(vmin[1]), float(vmax[2]))
prof_df = mesh_field.sample_field_along_line(geom, cp_stats["mean"], p1, p2, n=40)
fig, ax = plot_config.new_axes(
    xlabel="mean Cp [-]", ylabel="z [m]", title="Front-facade mean Cp profile"
)
ax.plot(prof_df["value"], prof_df["z"], "-o", ms=3)
dbg.savefig(fig, "front_facade_cp_profile.png", deliverable=True)
plot_config.close(fig)
dbg.save_csv(prof_df, "front_facade_cp_profile.csv", deliverable=True)
print("facade profile sampled:", prof_df.shape)